# PMPML Bus Network Analysis
Analysis of Pune's public bus network using PMPML's official published GTFS schedule data, examining service concentration by hour, corridor demand, and stop-level accessibility gaps.

**Data Source:** PMPML GTFS feed (`croyla/pmpml-gtfs`), representing the official published schedule — not real-time/actual operational data.

In [ ]:
import pandas as pd
import numpy as np
import geopy
print(geopy.__version__ )

In [ ]:
stops = pd.read_csv("stops.txt", dtype={"stop_id": str})
stop_time = pd.read_csv("stop_times.txt", dtype={"stop_id": str})
trip = pd.read_csv("trips.txt", dtype={'route_id': str})
routes = pd.read_csv("routes.txt", dtype={'route_id': str})

In [ ]:
print(stop_time.head(20))

In [ ]:
print(stop_time.sort_values(['trip_id', 'stop_sequence']).head(40))

# Inshight 1 - Hourly Trip Demand 

### 1 - Extracting starting stops of all trips

In [ ]:
first_stop = stop_time[stop_time['stop_sequence'] == 1].copy()
print(first_stop.head(60))

In [ ]:
print('Total unique trips', first_stop['trip_id'].nunique())
print("total rows in first_stop", len(first_stop))
print("Total unique first_stop, stop_id", first_stop['stop_id'].nunique())
print(first_stop['stop_id'].value_counts().head(20))

### Converting time into hour by splitting into parts based on ':' sign

In [ ]:
print(first_stop['arrival_time'].head())
first_stop['hour'] = first_stop['arrival_time'].str.split(':').str[0]
print("hour in first_stops")
print(first_stop['hour'])


In [ ]:
print(first_stop['hour'].value_counts())

### Counting each hour's trip counts

In [ ]:
hourly_counts = first_stop['hour'].value_counts().sort_index()
hourly_percentage = ((hourly_counts/hourly_counts.sum())*100).round(2)
print(hourly_percentage)

# Inshight 1 Conclusion
### Concentrates service around the morning commute — hours 8-10 AM alone account for ~20.5% of all scheduled trips citywide, despite being only 3 of the ~19 operating hours. Evening hours (17-19) show a smaller secondary bump (~15.9% combined), suggesting weaker evening commute coverage relative to morning

In [ ]:
hourly_percentage.to_csv("Hourly_trip_percentage.csv")

In [ ]:
hourly_summary = hourly_percentage.reset_index()
hourly_summary.columns = ['hour', 'percentage_of_trips']
hourly_summary.to_csv("Hourly_trip_percentage2.csv", index=False)

# Insight - 2 Finding the bussiest corridors

In [ ]:
route_counts = trip['route_id'].value_counts()


print(trip['route_id'].nunique())
print(route_counts.head(20))


In [ ]:
print(routes['route_type'].nunique())
print(trip['trip_id'].nunique())
print(len(trip))

In [ ]:
route_counts_named = route_counts.reset_index()
route_counts_named.columns = ['route', 'trips']


In [ ]:
print(route_counts_named.columns)
print(routes.columns)

In [ ]:
route_counts_named = route_counts_named.rename(columns={'route': 'route_id'})
print(route_counts_named.columns)
print(trip.columns)

### For this we will be using 'route_id' from 'trip' dataframe and 'route_long_name' from 'routes' dataframe

In [ ]:
route_counts_named = route_counts_named.merge(routes[['route_id', 'route_long_name']], on='route_id', how='left')

In [ ]:

print(route_counts_named.head(20))

In [ ]:
print(route_counts_named['route_long_name'].isna().sum())

#### From 'name' cloumn we will remove 'UP' and 'Down' indicating trips direction and sort them alphabetically to make trips all  bidirectional

In [ ]:
def corridor_noramlize(name):
    name = name.replace(' (UP)', '').replace(' (DOWN)', '')
    parts = name.split(' to ')
    if len(parts) == 2:
        return '|'.join(sorted(parts))
    return name

In [ ]:
route_counts_named['corridor'] = route_counts_named['route_long_name'].apply(corridor_noramlize)
corridor_trips = route_counts_named.groupby('corridor')['trips'].sum().sort_values(ascending=False)


## Insight 2 — Conclusion
Pune Station Moledina Stand ↔ Swargate leads at 300 trips/day, served by two overlapping route pairs (Route 5 + Route 6, both directions) rather than a single route. This sits in sharp contrast to the median corridor, which gets just 14 trips/day — showing how concentrated service is on a handful of trunk corridors.

In [ ]:
corridor_trips = corridor_trips.reset_index()
corridor_trips.columns = ['corridor_name', 'Trips']

In [ ]:
corridor_trips.to_csv("corridor_trips3.csv", index=False)

# Insight 3 — Underserved Stops Near Key Attractions
### Goal: identify bus stops with very low service frequency that sit close to high-need public locations (hospitals, colleges, universities, marketplaces)

In [ ]:
stop_visit_counts = stop_time['stop_id'].value_counts()
print(stop_visit_counts.describe())
print(stop_visit_counts.head(10))

### 3.1 — Counting total stop visits and attaching stop names/coordinates

In [ ]:
stop_visit_df = stop_visit_counts.reset_index()
stop_visit_df.columns = ['stop_id', 'total_trips']

stop_visit_df['stop_id'] = stop_visit_df['stop_id'].astype(str)
stops['stop_id'] = stops['stop_id'].astype(str)

stop_visit_df = stop_visit_df.merge(stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')
print(stop_visit_df.head(10))
print(stop_visit_df.shape)

In [ ]:
print(stop_visit_df['stop_id'].nunique())
print(stop_visit_df['stop_name'].isna().sum())

In [ ]:
print(stop_visit_df['total_trips'].describe())

### 3.2 — Defining "underserved" as the bottom 25th percentile of stops by total_trips (≤10 trips/day)

In [ ]:
underserved = stop_visit_df[stop_visit_df['total_trips'] <= 10]
print(underserved.shape)
print(underserved.sort_values('total_trips').head(10))

### 3.3 — Pulling real Pune POI locations (hospitals, colleges, universities, marketplaces) via the Overpass API


In [ ]:
import requests

overpass_url = "https://overpass.kumi.systems/api/interpreter"

headers = {
    "User-Agent": "PMPML-student-project/1.0"
}


overpass_query = """
[out:json][timeout:60];
area["name"="Pune"]["boundary"="administrative"]->.searchArea;
(
  node["amenity"="hospital"](area.searchArea);
  node["amenity"="college"](area.searchArea);
  node["amenity"="university"](area.searchArea);
  node["amenity"="marketplace"](area.searchArea);
);
out center;
"""

response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
print(response.status_code)
print(response.text[:300])
data = response.json()
print(len(data['elements']))

In [ ]:
print(response.text[:500])

In [ ]:
overpass_url = "https://overpass-api.de/api/interpreter"

headers = {
    "User-Agent": "PMPML-student-project/1.0",
    "Accept": "*/*",
    "Content-Type": "application/x-www-form-urlencoded"
}

response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
print(response.status_code)
print(response.text[:300])

In [ ]:
data = response.json()
print(len(data['elements']))
print(data['elements'][0])

In [ ]:
poi_list = []
for element in data['elements']:
    tags = element.get('tags', {})
    poi_list.append({
        'name': tags.get('name', 'Unknown'),
        'amenity': tags.get('amenity', 'Unknown'),
        'lat': element['lat'],
        'lon': element['lon']
    })

poi_df = pd.DataFrame(poi_list)
print(poi_df.shape)
print(poi_df['amenity'].value_counts())
print(poi_df.head(10))

### 3.4 — For each underserved stop, finding the nearest POI within 500m (name, type, distance)

In [ ]:

from geopy.distance import geodesic

def find_nearest_poi(stop_lat, stop_lon, poi_df, radius=500):
    nearest_name = None
    nearest_type = None
    min_dist = radius  # only consider within 500m

    for _, poi in poi_df.iterrows():
        dist = geodesic((stop_lat, stop_lon), (poi['lat'], poi['lon'])).meters
        if dist < min_dist:
            min_dist = dist
            nearest_name = poi.get('name', 'Unnamed')
            nearest_type = poi['amenity']  # matches your actual poi_df column

    if nearest_name is not None:
        return pd.Series([nearest_name, nearest_type, round(min_dist, 1)])
    else:
        return pd.Series([None, None, None])

# Run on all underserved stops (bottom 25th percentile, total_trips <= 10)
underserved = stop_visit_df[stop_visit_df['total_trips'] <= 10].copy()

underserved[['nearest_poi_name', 'nearest_poi_type', 'nearest_poi_distance_m']] = underserved.apply(
    lambda row: find_nearest_poi(row['stop_lat'], row['stop_lon'], poi_df),
    axis=1
)

# Keep only the ones that actually have a nearby POI (within 500m)
near_attraction_stops = underserved[underserved['nearest_poi_name'].notna()].copy()

print(near_attraction_stops.shape)
near_attraction_stops.sort_values('total_trips').head(10)

## Insight 3 — Conclusion
273 of 1,822 underserved stops (~15%) sit within 500m of a hospital, college, university, or marketplace — meaning a meaningful share of Pune's least-served stops are located right next to places with high, consistent foot traffic. Example: a stop near Sangavi Multispecialist Hospital gets just 2 trips/day despite sitting 7.6m from the hospital gate.

In [ ]:
print(near_attraction_stops.shape)
near_attraction_stops.to_csv("underserved_stops_near_attractions.csv", index=False)